### Set up 

In [55]:
# ==============================================================================
# CELLULE 1 : SETUP ET CHARGEMENT (INCLUANT LE ROYAUME-UNI)
# ==============================================================================
import pandas as pd
import numpy as np
import pandas_datareader.data as web
from datetime import datetime
from pathlib import Path
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from statsmodels.stats.diagnostic import acorr_ljungbox
from group_lasso import GroupLasso
from sklearn.model_selection import KFold
import warnings

warnings.filterwarnings("ignore")

# --- PARAMÈTRES GLOBAUX ---
START_DATE = datetime(2013, 1, 1) 
END_DATE = datetime(2026, 8, 1)
MIN_INDEX_DATE = '2015-03-01'

# --- CHARGEMENT GDELT ---
dir_geo = Path("./indicators_geo_monthly") 
dir_geo_uk = Path("./indicators_geo_monthly_UK")

# 1. Chargement France / US
if dir_geo.exists():
    files_geo = list(dir_geo.glob("*.parquet"))
    df_geo_main = pd.concat([pd.read_parquet(f) for f in files_geo], ignore_index=True)
else:
    raise FileNotFoundError("⚠ Dossier ./indicators_geo_monthly introuvable.")

# 2. Chargement UK
if dir_geo_uk.exists():
    files_uk = list(dir_geo_uk.glob("*.parquet"))
    if len(files_uk) > 0:
        df_geo_uk = pd.concat([pd.read_parquet(f) for f in files_uk], ignore_index=True)
        # On force la colonne region_key à 'UK' pour correspondre au dictionnaire
        df_geo_uk['region_key'] = 'UK' 
    else:
        print("⚠ Le dossier UK existe mais est vide (aucun fichier .parquet).")
        df_geo_uk = pd.DataFrame()
else:
    print(f"⚠ Dossier {dir_geo_uk} introuvable. UK ignoré.")
    df_geo_uk = pd.DataFrame()

# 3. Fusion finale
df_geo = pd.concat([df_geo_main, df_geo_uk], ignore_index=True)

# S'assurer que region_key existe bien avant le groupby
if 'region_key' in df_geo.columns:
    df_geo = df_geo.groupby(['period', 'region_key']).max().reset_index().copy()
    df_geo['period'] = pd.to_datetime(df_geo['period'])
    print(f"✓ Base RÉGIONALE GDELT chargée.")
    print(f"✓ Régions trouvées en mémoire : {df_geo['region_key'].unique().tolist()}")
    print(f"✓ Lignes totales : {len(df_geo)}")
else:
    print("⚠ ERREUR CRITIQUE : La colonne 'region_key' est absente des fichiers Parquet.")

✓ Base RÉGIONALE GDELT chargée.
✓ Régions trouvées en mémoire : ['Africa', 'China', 'European Union', 'France', 'India', 'Japan', 'Lebanon', 'Middle East', 'North America', 'Russia', 'South America', 'South East Asia', 'UK', 'US']
✓ Lignes totales : 1918


In [85]:
# ==============================================================================
# CELLULE 2 : FONCTIONS DE PRÉPARATION (ÉTAPES 0, 1, 2)
# ==============================================================================

def get_filtered_gdelt_cols(df):
    """ÉTAPE 0 : Exclut les variables mères (colinéarité parfaite)."""
    parents_to_exclude = [
        'att_weight_agriculture', 'att_weight_commodities', 'att_weight_energy',
        'att_weight_finance', 'att_weight_industry', 'att_weight_real_estate',
        'att_weight_tech', 'att_weight_transport'
    ]
    return [col for col in df.columns if str(col).startswith('att_weight_') and col not in parents_to_exclude]

def strict_stationarize(df):
    """
    ÉTAPE 1 : Test ADF -> Différence si besoin -> Winsorisation si choc extrême.
    """
    df_stat = df.copy()
    diff_count = 0
    
    for col in df_stat.columns:
        valid_data = df_stat[col].dropna()
        if len(valid_data) > 10:
            # 1. Premier test
            if adfuller(valid_data, autolag='AIC')[1] >= 0.05:
                # 2. Différenciation
                df_stat[col] = df_stat[col].diff()
                diff_count += 1
                
                # 3. Re-test
                valid_data_diff = df_stat[col].dropna()
                if len(valid_data_diff) > 10 and adfuller(valid_data_diff, autolag='AIC')[1] >= 0.05:
                    
                    # 4. LE SAUVETAGE : Winsorisation (Plafonnement au 95ème centile)
                    # On rabote le "giga pic" pour calmer la variance
                    p95 = valid_data_diff.quantile(0.95)
                    p05 = valid_data_diff.quantile(0.05)
                    df_stat[col] = df_stat[col].clip(lower=p05, upper=p95)
                    
                    print(f"  ⚕️ Sauvetage par Winsorisation appliqué sur : {col}")
                    
    return df_stat.dropna(), diff_count

# def fetch_and_prep_macro_dynamic(region_config, start_date, end_date, region=None):
#     """Télécharge la macroéconomie dynamique via FRED."""
#     macro_config = region_config['macro_vars']
    
#     # 1. Isoler les tickers FRED (avec gestion de la BCE pour la France si besoin)
#     fred_tickers = {}
#     for var_name, info in macro_config.items():
#         if region == 'France' and var_name == 'money':
#             continue  # Géré via la BCE
#         fred_tickers[info['ticker']] = var_name
        
#     df = web.DataReader(list(fred_tickers.keys()), 'fred', start_date, end_date)
#     df = df.rename(columns=fred_tickers).resample('MS').mean()
    
#     # 2. Branchement BCE pour la France
#     if region == 'France' and 'money' in macro_config:
#         ecb_url = "https://data-api.ecb.europa.eu/service/data/BSI/M.U2.Y.V.M30.X.1.U2.2300.Z01.E?format=csvdata"
#         df_ecb = pd.read_csv(ecb_url)
#         df_ecb['period'] = pd.to_datetime(df_ecb['TIME_PERIOD'])
#         df_ecb = df_ecb.set_index('period')
#         df_ecb = df_ecb.rename(columns={'OBS_VALUE': 'money'})
#         df_ecb = df_ecb[['money']].resample('MS').mean()
#         df = df.join(df_ecb, how='left')
        
#     # 3. Transformations mathématiques
#     final_cols = []
#     for var_name, info in macro_config.items():
#         if info['transform'] == 'pct_change':
#             df[var_name] = df[var_name].pct_change() * 100
#         elif info['transform'] == 'diff':
#             df[var_name] = df[var_name].diff()
#         final_cols.append(var_name)
        
#     for lag in region_config.get('inflation_lags', []):
#         lag_name = f'inflation_lag{lag}'
#         df[lag_name] = df['inflation'].shift(lag)
#         final_cols.append(lag_name)
        
#     return df[final_cols].dropna()

import io
import requests
import io
import requests
import pandas as pd
import pandas_datareader.data as web
import io
import requests
import csv
import pandas as pd
import pandas_datareader.data as web
import os
import pandas as pd
import pandas_datareader.data as web

def fetch_and_prep_macro_dynamic(region_config, start_date, end_date, region=None):
    """Télécharge la macroéconomie via FRED/BCE et lit les fichiers locaux pour la BoE."""
    macro_config = region_config['macro_vars']
    
    # 1. Trier les sources (FRED vs Fichiers Locaux)
    fred_tickers = {}
    local_tickers = {}
    
    for var_name, info in macro_config.items():
        if region == 'France' and var_name == 'money':
            continue  
        if region == 'UK' and info['ticker'] in ['IUMABEDR', 'LPMBD93', 'XUDLUSS']:
            local_tickers[info['ticker']] = var_name
            continue
            
        fred_tickers[info['ticker']] = var_name
        
    # --- A. CHARGEMENT FRED ---
    df = pd.DataFrame()
    if fred_tickers:
        df = web.DataReader(list(fred_tickers.keys()), 'fred', start_date, end_date)
        df = df.rename(columns=fred_tickers).resample('MS').mean()
        
    # --- B. CHARGEMENT API BCE (FRANCE) ---
    if region == 'France' and 'money' in macro_config:
        ecb_url = "https://data-api.ecb.europa.eu/service/data/BSI/M.U2.Y.V.M30.X.1.U2.2300.Z01.E?format=csvdata"
        try:
            df_ecb = pd.read_csv(ecb_url)
            df_ecb['period'] = pd.to_datetime(df_ecb['TIME_PERIOD'])
            df_ecb = df_ecb.set_index('period')
            df_ecb = df_ecb.rename(columns={'OBS_VALUE': 'money'})
            df_ecb = df_ecb[['money']].resample('MS').mean()
            df = df.join(df_ecb, how='left') if not df.empty else df_ecb
        except Exception as e:
            print(f"⚠ Erreur BCE: {e}")
            
    # --- C. CHARGEMENT FICHIERS LOCAUX BoE (UK) ---
    if region == 'UK' and local_tickers:
        df_local_all = pd.DataFrame()
        
        for ticker, var_name in local_tickers.items():
            file_path = f"data/BoE/{ticker}.csv"
            
            if os.path.exists(file_path):
                # Lecture robuste (gestion des encodages potentiels de la BoE)
                try:
                    df_temp = pd.read_csv(file_path, encoding='utf-8')
                except UnicodeDecodeError:
                    df_temp = pd.read_csv(file_path, encoding='ISO-8859-1')
                
                date_col = df_temp.columns[0]
                
                # Conversion des dates type "10 Aug 26"
                df_temp['period'] = pd.to_datetime(df_temp[date_col], errors='coerce')
                df_temp = df_temp.dropna(subset=['period']).set_index('period')
                
                # Recherche dynamique de la colonne de valeur
                val_col = None
                for c in df_temp.columns:
                    if ticker in c:
                        val_col = c
                        break
                
                if val_col:
                    # Nettoyage : retirer virgules et espaces cachés
                    if df_temp[val_col].dtype == object:
                        df_temp[val_col] = df_temp[val_col].astype(str).str.replace(',', '').str.strip()
                    df_temp[val_col] = pd.to_numeric(df_temp[val_col], errors='coerce')
                    
                    # Le resample('MS').mean() est crucial ici pour moyenner les taux quotidiens (XUDLUSS)
                    series = df_temp[[val_col]].rename(columns={val_col: var_name}).resample('MS').mean()
                    
                    if df_local_all.empty:
                        df_local_all = series
                    else:
                        df_local_all = df_local_all.join(series, how='outer')
            else:
                print(f"  -> ⚠ Fichier local introuvable : {file_path}")
                
        if not df_local_all.empty:
            # On utilise un outer join pour s'assurer que les dates coïncident parfaitement avec FRED
            df = df.join(df_local_all, how='outer') if not df.empty else df_local_all

    # --- 4. TRANSFORMATIONS MATHÉMATIQUES ---
    final_cols = []
    for var_name, info in macro_config.items():
        if var_name not in df.columns:
            continue
        if info['transform'] == 'pct_change':
            df[var_name] = df[var_name].pct_change() * 100
        elif info['transform'] == 'diff':
            df[var_name] = df[var_name].diff()
        final_cols.append(var_name)
        
    for lag in region_config.get('inflation_lags', []):
        lag_name = f'inflation_lag{lag}'
        df[lag_name] = df['inflation'].shift(lag)
        final_cols.append(lag_name)
        
    # Ne garder que la fenêtre demandée
    df = df[(df.index >= start_date) & (df.index <= end_date)]
    
    # Isolation des variables finales
    df_final_cols = df[final_cols]
    df_clean = df_final_cols.dropna()
    
    # --- DÉTECTEUR DE CRASH ---
    if len(df_clean) == 0:
        print(f"\n" + "!"*60)
        print(f" ⚠ CRASH DÉTECTÉ POUR : {region.upper()}")
        print(f" La matrice est vide. Voici les coupables (Valeurs Manquantes) :")
        print(df_final_cols.isnull().sum())
        print("!"*60 + "\n")
        raise ValueError(f"Le pipeline de {region} s'est arrêté car des colonnes sont vides.")
        
    return df_clean


def apply_fwl_orthogonalization(X_macro, X_gdelt, y):
    """ÉTAPE 2 : Purge macroéconomique (Frisch-Waugh-Lovell)."""
    ols_y = LinearRegression(fit_intercept=True).fit(X_macro, y)
    y_tilde = y - ols_y.predict(X_macro)
    
    ols_X = LinearRegression(fit_intercept=True).fit(X_macro, X_gdelt)
    X_gdelt_tilde = pd.DataFrame(
        X_gdelt.values - ols_X.predict(X_macro), 
        columns=X_gdelt.columns, index=X_gdelt.index
    )
    return y_tilde, X_gdelt_tilde

In [86]:
# ==============================================================================
# CELLULE 3 : MOTEURS D'ESTIMATION (Yuan & Lin 2006 / Simon et al. 2013)
# ==============================================================================

def run_yuan_lin_group_lasso(X, y, groups, sectors_names):
    """
    ÉTAPE 3 : Méthode de Yuan et Lin (2006)
    Utilise le critère Cp approximatif sur un chemin de solutions.
    """
    n = len(y)
    unique_groups = np.unique(groups)
    
    # 1. Estimateur OLS complet pour calculer sigma^2 et les normes de référence
    ols = LinearRegression(fit_intercept=False).fit(X, y)
    beta_ols = ols.coef_
    sigma2 = np.mean((y - ols.predict(X))**2)
    
    # 2. Création du chemin de lambdas
    # On évalue sur 50 valeurs pour trouver le minimum de la courbe Cp
    lambdas = np.logspace(-3, 1, 50)
    
    best_cp = np.inf
    best_model = None
    best_beta = None
    best_lbd = None
    
    for lbd in lambdas:
        gl = GroupLasso(groups=groups, l1_reg=0.0, group_reg=lbd, 
                        fit_intercept=False, n_iter=5000, supress_warning=True)
        gl.fit(X.values, y.values.reshape(-1, 1))
        beta_gl = gl.coef_.flatten()
        preds = gl.predict(X.values).flatten()
        rss = np.sum((y - preds)**2)
        
        # 3. Calcul exact des degrés de libertés selon Yuan et Lin (2006)
        df_penalty = 0
        for g in unique_groups:
            idx = np.where(groups == g)[0]
            beta_g = beta_gl[idx]
            if np.linalg.norm(beta_g) > 0:
                beta_g_ols = beta_ols[idx]
                norm_ols = np.linalg.norm(beta_g_ols)
                # Formule : 1 + (||beta_g|| / ||beta_g_OLS||) * (pj - 1)
                df_penalty += 1 + (np.linalg.norm(beta_g) / norm_ols) * (len(idx) - 1)
                
        # 4. Calcul du Cp
        cp = (rss / sigma2) - n + 2 * df_penalty
        
        if cp < best_cp:
            best_cp = cp
            best_beta = beta_gl
            best_lbd = lbd
            
    coef_df = pd.DataFrame({'Variable': X.columns, 'Secteur': sectors_names, 'Coefficient': best_beta})
    active = coef_df[coef_df['Coefficient'] != 0].sort_values(by=['Secteur', 'Coefficient'])
    return active, best_lbd, best_cp

def run_simon_sparse_group_lasso(X, y, groups, sectors_names, alpha=0.05):
    """
    ÉTAPE 4 : Méthode de Simon et al. (2013) corrigée pour la macro.
    Utilise un KFold standard avec mélange pour éviter la panique des ruptures 
    structurelles récentes, respectant la logique d'échantillonnage de l'article d'origine.
    """
    # Remplacement du TimeSeriesSplit par un KFold avec mélange
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    
    lambdas = np.logspace(-3, 0, 100)
    best_mse = np.inf
    best_lbd = None
    
    for lbd in lambdas:
        l1_penalty = lbd * alpha
        group_penalty = lbd * (1 - alpha)
        
        sgl = GroupLasso(groups=groups, l1_reg=l1_penalty, group_reg=group_penalty,
                         fit_intercept=False, n_iter=3000, supress_warning=True)
        
        mse_scores = []
        for train_index, test_index in kf.split(X):
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train, y_test = y.iloc[train_index], y.iloc[test_index]
            
            sgl.fit(X_train.values, y_train.values.reshape(-1, 1))
            preds = sgl.predict(X_test.values).flatten()
            mse_scores.append(np.mean((y_test.values - preds)**2))
            
        avg_mse = np.mean(mse_scores)
        
        if avg_mse < best_mse:
            best_mse = avg_mse
            best_lbd = lbd
            
    final_sgl = GroupLasso(groups=groups, l1_reg=best_lbd*alpha, group_reg=best_lbd*(1-alpha),
                           fit_intercept=False, n_iter=5000, supress_warning=True)
    final_sgl.fit(X.values, y.values.reshape(-1, 1))
    
    coef_df = pd.DataFrame({'Variable': X.columns, 'Secteur': sectors_names, 'Coefficient': final_sgl.coef_.flatten()})
    active = coef_df[coef_df['Coefficient'] != 0].sort_values(by=['Secteur', 'Coefficient'])
    
    return active, best_lbd, alpha

In [92]:
# ==============================================================================
# CELLULE 4 : EXÉCUTION DU PIPELINE COMPLET (AVEC INFÉRENCE ÉTAPE 5)
# ==============================================================================

CONFIG_REGIONS = {
    'France': {
        'macro_vars': {
            'inflation': {'ticker': 'CP0000FRM086NEST', 'transform': 'pct_change'},
            'rate': {'ticker': 'IRSTCI01FRM156N', 'transform': 'diff'},
            'unemp': {'ticker': 'LRHUTTTTFRM156S', 'transform': 'diff'},
            'indpro': {'ticker': 'FRAPRINTO01GYSAM', 'transform': 'pct_change'},
            'money': {'ticker': 'BCE_M3_API', 'transform': 'none'}, 
            'fx': {'ticker': 'DEXUSEU', 'transform': 'pct_change'},
            'mich': {'ticker': 'CSINFT02FRM460S', 'transform': 'diff'},
            'oil': {'ticker': 'DCOILBRENTEU', 'transform': 'pct_change'}
        },
        'inflation_lags': [1, 2, 3, 12],
        'deseasonalize_y': True
    }
}

final_results = {}

for region, config in CONFIG_REGIONS.items():
    print(f"\n" + "="*70)
    print(f" PIPELINE ACADÉMIQUE COMPLET : {region.upper()}")
    print(f"======================================================================")
    
    # --- PRÉPARATION MACRO ---
    df_macro = fetch_and_prep_macro_dynamic(config, START_DATE, END_DATE, region=region)
    df_macro = df_macro[df_macro.index >= MIN_INDEX_DATE]
    
    # --- PRÉPARATION GDELT ---
    cols_gdelt = get_filtered_gdelt_cols(df_geo)
    df_gdelt_raw = df_geo[df_geo['region_key'] == region].set_index('period')[cols_gdelt]
    
    # ---------------------------------------------------------
    # AMPUTATION CHIRURGICALE : On retire la variable asymétrique
    # ---------------------------------------------------------
    col_to_drop = 'att_weight_finance_international_orgs'
    if col_to_drop in df_gdelt_raw.columns:
        df_gdelt_raw = df_gdelt_raw.drop(columns=[col_to_drop])
    
    # --- STATIONNARISATION ---
    df_gdelt_stat, diff_count = strict_stationarize(df_gdelt_raw)
    print(f"✓ ÉTAPE 0 & 1 : {diff_count} variables stationnarisées. Les variables mères sont exclues.")
    
    # --- JOINTURE ---
    df_final = df_gdelt_stat.join(df_macro, how='inner')
    y_raw = df_final['inflation']
    
    if config['deseasonalize_y']:
        decomp = seasonal_decompose(y_raw, model='additive', period=12)
        y_corr = (y_raw - decomp.seasonal).dropna()
    else:
        y_corr = y_raw.dropna()
        
    X_full = df_final.drop(columns=['inflation'])
    y_corr, X_contemp = y_corr.align(X_full, join='inner')
    
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X_contemp), columns=X_contemp.columns, index=X_contemp.index)
    
    X_macro = X_scaled[[col for col in X_scaled.columns if col not in cols_gdelt]]
    X_gdelt = X_scaled[[col for col in X_scaled.columns if col in cols_gdelt]]
    
    # --- FWL ---
    y_tilde, X_gdelt_tilde = apply_fwl_orthogonalization(X_macro, X_gdelt, y_corr)
    pval = acorr_ljungbox(y_tilde, lags=[12], return_df=True)['lb_pvalue'].values[0]
    print(f"✓ ÉTAPE 2 (FWL) : Matrice de {len(y_tilde)} mois. Ljung-Box p-value = {pval:.4f}")
    
    secteurs = [col.split('_')[2] for col in X_gdelt_tilde.columns]
    sector_to_id = {sec: i for i, sec in enumerate(list(set(secteurs)))}
    groups_array = np.array([sector_to_id[sec] for sec in secteurs])
    
    # --- ÉTAPE 3 : GROUP LASSO (YUAN & LIN) ---
    print("\n[ÉTAPE 3] Group Lasso (Méthode Yuan & Lin 2006, sélection via Cp)...")
    active_gl, lbd_gl, cp_gl = run_yuan_lin_group_lasso(X_gdelt_tilde, y_tilde, groups_array, secteurs)
    print(f"  -> Lambda optimal : {lbd_gl:.4f} (Critère Cp min : {cp_gl:.2f})")
    print(f"  -> {len(active_gl)} variables conservées.")
        
    # --- ÉTAPE 4 : SPARSE GROUP LASSO (SIMON ET AL.) ---
    print("\n[ÉTAPE 4] Sparse Group Lasso (Méthode Simon et al. 2013, sélection via CV avec KFold)...")
    active_sgl, lbd_sgl, alpha_sgl = run_simon_sparse_group_lasso(X_gdelt_tilde, y_tilde, groups_array, secteurs, alpha=0.05)
    print(f"  -> Lambda optimal : {lbd_sgl:.4f} (avec Alpha fixe à {alpha_sgl})")
    
    if active_sgl.empty:
        print("  -> Résultat : Aucune variable conservée.")
    else:
        print(f"  -> {len(active_sgl)} variables conservées.")
        
        # --- ÉTAPE 5 : INFÉRENCE POST-LASSO OLS ---
        print("\n[ÉTAPE 5] Tribunal Statistique : Inférence Post-Lasso OLS sur les survivants...")
        post_lasso_df, r2_sgl, r2_adj_sgl = run_post_lasso_ols(X_gdelt_tilde, y_tilde, active_sgl)
        
        print(f"  -> R-squared (Pouvoir explicatif des médias purs) : {r2_sgl:.4f}")
        print(f"  -> R-squared Ajusté : {r2_adj_sgl:.4f}\n")
        print(post_lasso_df.to_string(index=False))
        
        final_results[region] = {
            'SGL_Selection': active_sgl,
            'OLS_Inference': post_lasso_df,
            'R2_adj': r2_adj_sgl
        }


 PIPELINE ACADÉMIQUE COMPLET : FRANCE
✓ ÉTAPE 0 & 1 : 22 variables stationnarisées. Les variables mères sont exclues.
✓ ÉTAPE 2 (FWL) : Matrice de 122 mois. Ljung-Box p-value = 0.2963

[ÉTAPE 3] Group Lasso (Méthode Yuan & Lin 2006, sélection via Cp)...
  -> Lambda optimal : 0.0026 (Critère Cp min : 73.38)
  -> 50 variables conservées.

[ÉTAPE 4] Sparse Group Lasso (Méthode Simon et al. 2013, sélection via CV avec KFold)...
  -> Lambda optimal : 0.0087 (avec Alpha fixe à 0.05)
  -> 44 variables conservées.

[ÉTAPE 5] Tribunal Statistique : Inférence Post-Lasso OLS sur les survivants...
  -> R-squared (Pouvoir explicatif des médias purs) : 0.4896
  -> R-squared Ajusté : 0.1979

                                  Variable  Coefficient  P-value  CI_Lower  CI_Upper
                                 Intercept       0.0000   1.0000   -0.0457    0.0457
         att_weight_agriculture_regulation      -0.0539   0.0977   -0.1178    0.0101
        att_weight_agriculture_rural_labor      -0.0865   

In [95]:
# ==============================================================================
# CELLULE 4 : EXÉCUTION DU PIPELINE COMPLET (AVEC INFÉRENCE ÉTAPE 5)
# ==============================================================================

CONFIG_REGIONS = {
    'France': {
        'macro_vars': {
            'inflation': {'ticker': 'CP0000FRM086NEST', 'transform': 'pct_change'},
            'rate': {'ticker': 'IRSTCI01FRM156N', 'transform': 'diff'},
            'unemp': {'ticker': 'LRHUTTTTFRM156S', 'transform': 'diff'},
            'indpro': {'ticker': 'FRAPRINTO01GYSAM', 'transform': 'pct_change'},
            'money': {'ticker': 'BCE_M3_API', 'transform': 'none'}, 
            'fx': {'ticker': 'DEXUSEU', 'transform': 'pct_change'},
            'mich': {'ticker': 'CSINFT02FRM460S', 'transform': 'diff'},
            'oil': {'ticker': 'DCOILBRENTEU', 'transform': 'pct_change'}
        },
        'inflation_lags': [1, 2, 3, 12],
        'deseasonalize_y': False
    }
}

final_results = {}

for region, config in CONFIG_REGIONS.items():
    print(f"\n" + "="*70)
    print(f" PIPELINE ACADÉMIQUE COMPLET : {region.upper()}")
    print(f"======================================================================")
    
    # --- PRÉPARATION MACRO ---
    df_macro = fetch_and_prep_macro_dynamic(config, START_DATE, END_DATE, region=region)
    df_macro = df_macro[df_macro.index >= MIN_INDEX_DATE]
    
    # --- PRÉPARATION GDELT ---
    cols_gdelt = get_filtered_gdelt_cols(df_geo)
    df_gdelt_raw = df_geo[df_geo['region_key'] == region].set_index('period')[cols_gdelt]
    
    # ---------------------------------------------------------
    # AMPUTATION CHIRURGICALE : On retire la variable asymétrique
    # ---------------------------------------------------------
    col_to_drop = 'att_weight_finance_international_orgs'
    if col_to_drop in df_gdelt_raw.columns:
        df_gdelt_raw = df_gdelt_raw.drop(columns=[col_to_drop])
    
    # --- STATIONNARISATION ---
    df_gdelt_stat, diff_count = strict_stationarize(df_gdelt_raw)
    print(f"✓ ÉTAPE 0 & 1 : {diff_count} variables stationnarisées. Les variables mères sont exclues.")
    
    # --- JOINTURE ---
    df_final = df_gdelt_stat.join(df_macro, how='inner')
    y_raw = df_final['inflation']
    
    if config['deseasonalize_y']:
        decomp = seasonal_decompose(y_raw, model='additive', period=12)
        y_corr = (y_raw - decomp.seasonal).dropna()
    else:
        y_corr = y_raw.dropna()
        
    X_full = df_final.drop(columns=['inflation'])
    y_corr, X_contemp = y_corr.align(X_full, join='inner')
    
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X_contemp), columns=X_contemp.columns, index=X_contemp.index)
    
    X_macro = X_scaled[[col for col in X_scaled.columns if col not in cols_gdelt]]
    X_gdelt = X_scaled[[col for col in X_scaled.columns if col in cols_gdelt]]
    
    # --- FWL ---
    y_tilde, X_gdelt_tilde = apply_fwl_orthogonalization(X_macro, X_gdelt, y_corr)
    pval = acorr_ljungbox(y_tilde, lags=[12], return_df=True)['lb_pvalue'].values[0]
    print(f"✓ ÉTAPE 2 (FWL) : Matrice de {len(y_tilde)} mois. Ljung-Box p-value = {pval:.4f}")
    
    secteurs = [col.split('_')[2] for col in X_gdelt_tilde.columns]
    sector_to_id = {sec: i for i, sec in enumerate(list(set(secteurs)))}
    groups_array = np.array([sector_to_id[sec] for sec in secteurs])
    
    # --- ÉTAPE 3 : GROUP LASSO (YUAN & LIN) ---
    print("\n[ÉTAPE 3] Group Lasso (Méthode Yuan & Lin 2006, sélection via Cp)...")
    active_gl, lbd_gl, cp_gl = run_yuan_lin_group_lasso(X_gdelt_tilde, y_tilde, groups_array, secteurs)
    print(f"  -> Lambda optimal : {lbd_gl:.4f} (Critère Cp min : {cp_gl:.2f})")
    print(f"  -> {len(active_gl)} variables conservées.")
        
    # --- ÉTAPE 4 : SPARSE GROUP LASSO (SIMON ET AL.) ---
    print("\n[ÉTAPE 4] Sparse Group Lasso (Méthode Simon et al. 2013, sélection via CV avec KFold)...")
    active_sgl, lbd_sgl, alpha_sgl = run_simon_sparse_group_lasso(X_gdelt_tilde, y_tilde, groups_array, secteurs, alpha=0.05)
    print(f"  -> Lambda optimal : {lbd_sgl:.4f} (avec Alpha fixe à {alpha_sgl})")
    
    if active_sgl.empty:
        print("  -> Résultat : Aucune variable conservée.")
    else:
        print(f"  -> {len(active_sgl)} variables conservées.")
        
        # --- ÉTAPE 5 : INFÉRENCE POST-LASSO OLS ---
        print("\n[ÉTAPE 5] Tribunal Statistique : Inférence Post-Lasso OLS sur les survivants...")
        post_lasso_df, r2_sgl, r2_adj_sgl = run_post_lasso_ols(X_gdelt_tilde, y_tilde, active_sgl)
        
        print(f"  -> R-squared (Pouvoir explicatif des médias purs) : {r2_sgl:.4f}")
        print(f"  -> R-squared Ajusté : {r2_adj_sgl:.4f}\n")
        print(post_lasso_df.to_string(index=False))
        
        final_results[region] = {
            'SGL_Selection': active_sgl,
            'OLS_Inference': post_lasso_df,
            'R2_adj': r2_adj_sgl
        }


 PIPELINE ACADÉMIQUE COMPLET : FRANCE
✓ ÉTAPE 0 & 1 : 22 variables stationnarisées. Les variables mères sont exclues.
✓ ÉTAPE 2 (FWL) : Matrice de 122 mois. Ljung-Box p-value = 0.2136

[ÉTAPE 3] Group Lasso (Méthode Yuan & Lin 2006, sélection via Cp)...
  -> Lambda optimal : 0.0031 (Critère Cp min : 75.68)
  -> 50 variables conservées.

[ÉTAPE 4] Sparse Group Lasso (Méthode Simon et al. 2013, sélection via CV avec KFold)...
  -> Lambda optimal : 0.0076 (avec Alpha fixe à 0.05)
  -> 45 variables conservées.

[ÉTAPE 5] Tribunal Statistique : Inférence Post-Lasso OLS sur les survivants...
  -> R-squared (Pouvoir explicatif des médias purs) : 0.5030
  -> R-squared Ajusté : 0.2088

                                  Variable  Coefficient  P-value  CI_Lower  CI_Upper
                                 Intercept      -0.0000   1.0000   -0.0454    0.0454
         att_weight_agriculture_regulation      -0.0740   0.0230   -0.1376   -0.0105
        att_weight_agriculture_rural_labor      -0.1329   

In [96]:
# ==============================================================================
# CELLULE 4 : EXÉCUTION DU PIPELINE COMPLET (AVEC INFÉRENCE ÉTAPE 5)
# ==============================================================================

CONFIG_REGIONS = {
    'France': {
        'macro_vars': {
            'inflation': {'ticker': 'CP0000FRM086NEST', 'transform': 'pct_change'},
            'rate': {'ticker': 'IRSTCI01FRM156N', 'transform': 'diff'},
            'unemp': {'ticker': 'LRHUTTTTFRM156S', 'transform': 'diff'},
            'indpro': {'ticker': 'FRAPRINTO01GYSAM', 'transform': 'pct_change'},
            'money': {'ticker': 'BCE_M3_API', 'transform': 'none'}, 
            'fx': {'ticker': 'DEXUSEU', 'transform': 'pct_change'},
            'mich': {'ticker': 'CSINFT02FRM460S', 'transform': 'diff'},
            'oil': {'ticker': 'DCOILBRENTEU', 'transform': 'pct_change'}
        },
        'inflation_lags': [1, 2, 3, 6, 12],
        'deseasonalize_y': False
    }
}

final_results = {}

for region, config in CONFIG_REGIONS.items():
    print(f"\n" + "="*70)
    print(f" PIPELINE ACADÉMIQUE COMPLET : {region.upper()}")
    print(f"======================================================================")
    
    # --- PRÉPARATION MACRO ---
    df_macro = fetch_and_prep_macro_dynamic(config, START_DATE, END_DATE, region=region)
    df_macro = df_macro[df_macro.index >= MIN_INDEX_DATE]
    
    # --- PRÉPARATION GDELT ---
    cols_gdelt = get_filtered_gdelt_cols(df_geo)
    df_gdelt_raw = df_geo[df_geo['region_key'] == region].set_index('period')[cols_gdelt]
    
    # ---------------------------------------------------------
    # AMPUTATION CHIRURGICALE : On retire la variable asymétrique
    # ---------------------------------------------------------
    col_to_drop = 'att_weight_finance_international_orgs'
    if col_to_drop in df_gdelt_raw.columns:
        df_gdelt_raw = df_gdelt_raw.drop(columns=[col_to_drop])
    
    # --- STATIONNARISATION ---
    df_gdelt_stat, diff_count = strict_stationarize(df_gdelt_raw)
    print(f"✓ ÉTAPE 0 & 1 : {diff_count} variables stationnarisées. Les variables mères sont exclues.")
    
    # --- JOINTURE ---
    df_final = df_gdelt_stat.join(df_macro, how='inner')
    y_raw = df_final['inflation']
    
    if config['deseasonalize_y']:
        decomp = seasonal_decompose(y_raw, model='additive', period=12)
        y_corr = (y_raw - decomp.seasonal).dropna()
    else:
        y_corr = y_raw.dropna()
        
    X_full = df_final.drop(columns=['inflation'])
    y_corr, X_contemp = y_corr.align(X_full, join='inner')
    
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X_contemp), columns=X_contemp.columns, index=X_contemp.index)
    
    X_macro = X_scaled[[col for col in X_scaled.columns if col not in cols_gdelt]]
    X_gdelt = X_scaled[[col for col in X_scaled.columns if col in cols_gdelt]]
    
    # --- FWL ---
    y_tilde, X_gdelt_tilde = apply_fwl_orthogonalization(X_macro, X_gdelt, y_corr)
    pval = acorr_ljungbox(y_tilde, lags=[12], return_df=True)['lb_pvalue'].values[0]
    print(f"✓ ÉTAPE 2 (FWL) : Matrice de {len(y_tilde)} mois. Ljung-Box p-value = {pval:.4f}")
    
    secteurs = [col.split('_')[2] for col in X_gdelt_tilde.columns]
    sector_to_id = {sec: i for i, sec in enumerate(list(set(secteurs)))}
    groups_array = np.array([sector_to_id[sec] for sec in secteurs])
    
    # --- ÉTAPE 3 : GROUP LASSO (YUAN & LIN) ---
    print("\n[ÉTAPE 3] Group Lasso (Méthode Yuan & Lin 2006, sélection via Cp)...")
    active_gl, lbd_gl, cp_gl = run_yuan_lin_group_lasso(X_gdelt_tilde, y_tilde, groups_array, secteurs)
    print(f"  -> Lambda optimal : {lbd_gl:.4f} (Critère Cp min : {cp_gl:.2f})")
    print(f"  -> {len(active_gl)} variables conservées.")
        
    # --- ÉTAPE 4 : SPARSE GROUP LASSO (SIMON ET AL.) ---
    print("\n[ÉTAPE 4] Sparse Group Lasso (Méthode Simon et al. 2013, sélection via CV avec KFold)...")
    active_sgl, lbd_sgl, alpha_sgl = run_simon_sparse_group_lasso(X_gdelt_tilde, y_tilde, groups_array, secteurs, alpha=0.05)
    print(f"  -> Lambda optimal : {lbd_sgl:.4f} (avec Alpha fixe à {alpha_sgl})")
    
    if active_sgl.empty:
        print("  -> Résultat : Aucune variable conservée.")
    else:
        print(f"  -> {len(active_sgl)} variables conservées.")
        
        # --- ÉTAPE 5 : INFÉRENCE POST-LASSO OLS ---
        print("\n[ÉTAPE 5] Tribunal Statistique : Inférence Post-Lasso OLS sur les survivants...")
        post_lasso_df, r2_sgl, r2_adj_sgl = run_post_lasso_ols(X_gdelt_tilde, y_tilde, active_sgl)
        
        print(f"  -> R-squared (Pouvoir explicatif des médias purs) : {r2_sgl:.4f}")
        print(f"  -> R-squared Ajusté : {r2_adj_sgl:.4f}\n")
        print(post_lasso_df.to_string(index=False))
        
        final_results[region] = {
            'SGL_Selection': active_sgl,
            'OLS_Inference': post_lasso_df,
            'R2_adj': r2_adj_sgl
        }


 PIPELINE ACADÉMIQUE COMPLET : FRANCE
✓ ÉTAPE 0 & 1 : 22 variables stationnarisées. Les variables mères sont exclues.
✓ ÉTAPE 2 (FWL) : Matrice de 122 mois. Ljung-Box p-value = 0.2740

[ÉTAPE 3] Group Lasso (Méthode Yuan & Lin 2006, sélection via Cp)...
  -> Lambda optimal : 0.0031 (Critère Cp min : 74.19)
  -> 50 variables conservées.

[ÉTAPE 4] Sparse Group Lasso (Méthode Simon et al. 2013, sélection via CV avec KFold)...
  -> Lambda optimal : 0.0066 (avec Alpha fixe à 0.05)
  -> 45 variables conservées.

[ÉTAPE 5] Tribunal Statistique : Inférence Post-Lasso OLS sur les survivants...
  -> R-squared (Pouvoir explicatif des médias purs) : 0.5074
  -> R-squared Ajusté : 0.2158

                                  Variable  Coefficient  P-value  CI_Lower  CI_Upper
                                 Intercept       0.0000   1.0000   -0.0449    0.0449
         att_weight_agriculture_regulation      -0.0851   0.0118   -0.1507   -0.0194
        att_weight_agriculture_rural_labor      -0.1267   

In [93]:
# ==============================================================================
# CELLULE 4 : EXÉCUTION DU PIPELINE COMPLET (AVEC INFÉRENCE ÉTAPE 5)
# ==============================================================================

CONFIG_REGIONS = {
    'US': {
        'macro_vars': {
            'inflation': {'ticker': 'CPIAUCSL', 'transform': 'pct_change'},
            'rate': {'ticker': 'FEDFUNDS', 'transform': 'diff'},
            'unemp': {'ticker': 'UNRATE', 'transform': 'diff'},
            'indpro': {'ticker': 'INDPRO', 'transform': 'pct_change'},
            'money': {'ticker': 'M2SL', 'transform': 'pct_change'}, 
            'fx': {'ticker': 'DEXUSEU', 'transform': 'pct_change'},
            'mich': {'ticker': 'MICH', 'transform': 'diff'},
            'oil': {'ticker': 'WTISPLC', 'transform': 'pct_change'}
        },
        'inflation_lags': [1, 2, 3],
        'deseasonalize_y': False
    }
}

final_results = {}

for region, config in CONFIG_REGIONS.items():
    print(f"\n" + "="*70)
    print(f" PIPELINE ACADÉMIQUE COMPLET : {region.upper()}")
    print(f"======================================================================")
    
    # --- PRÉPARATION MACRO ---
    df_macro = fetch_and_prep_macro_dynamic(config, START_DATE, END_DATE, region=region)
    df_macro = df_macro[df_macro.index >= MIN_INDEX_DATE]
    
    # --- PRÉPARATION GDELT ---
    cols_gdelt = get_filtered_gdelt_cols(df_geo)
    df_gdelt_raw = df_geo[df_geo['region_key'] == region].set_index('period')[cols_gdelt]
    
    # ---------------------------------------------------------
    # AMPUTATION CHIRURGICALE : On retire la variable asymétrique
    # ---------------------------------------------------------
    col_to_drop = 'att_weight_finance_international_orgs'
    if col_to_drop in df_gdelt_raw.columns:
        df_gdelt_raw = df_gdelt_raw.drop(columns=[col_to_drop])
    
    # --- STATIONNARISATION ---
    df_gdelt_stat, diff_count = strict_stationarize(df_gdelt_raw)
    print(f"✓ ÉTAPE 0 & 1 : {diff_count} variables stationnarisées. Les variables mères sont exclues.")
    
    # --- JOINTURE ---
    df_final = df_gdelt_stat.join(df_macro, how='inner')
    y_raw = df_final['inflation']
    
    if config['deseasonalize_y']:
        decomp = seasonal_decompose(y_raw, model='additive', period=12)
        y_corr = (y_raw - decomp.seasonal).dropna()
    else:
        y_corr = y_raw.dropna()
        
    X_full = df_final.drop(columns=['inflation'])
    y_corr, X_contemp = y_corr.align(X_full, join='inner')
    
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X_contemp), columns=X_contemp.columns, index=X_contemp.index)
    
    X_macro = X_scaled[[col for col in X_scaled.columns if col not in cols_gdelt]]
    X_gdelt = X_scaled[[col for col in X_scaled.columns if col in cols_gdelt]]
    
    # --- FWL ---
    y_tilde, X_gdelt_tilde = apply_fwl_orthogonalization(X_macro, X_gdelt, y_corr)
    pval = acorr_ljungbox(y_tilde, lags=[12], return_df=True)['lb_pvalue'].values[0]
    print(f"✓ ÉTAPE 2 (FWL) : Matrice de {len(y_tilde)} mois. Ljung-Box p-value = {pval:.4f}")
    
    secteurs = [col.split('_')[2] for col in X_gdelt_tilde.columns]
    sector_to_id = {sec: i for i, sec in enumerate(list(set(secteurs)))}
    groups_array = np.array([sector_to_id[sec] for sec in secteurs])
    
    # --- ÉTAPE 3 : GROUP LASSO (YUAN & LIN) ---
    print("\n[ÉTAPE 3] Group Lasso (Méthode Yuan & Lin 2006, sélection via Cp)...")
    active_gl, lbd_gl, cp_gl = run_yuan_lin_group_lasso(X_gdelt_tilde, y_tilde, groups_array, secteurs)
    print(f"  -> Lambda optimal : {lbd_gl:.4f} (Critère Cp min : {cp_gl:.2f})")
    print(f"  -> {len(active_gl)} variables conservées.")
        
    # --- ÉTAPE 4 : SPARSE GROUP LASSO (SIMON ET AL.) ---
    print("\n[ÉTAPE 4] Sparse Group Lasso (Méthode Simon et al. 2013, sélection via CV avec KFold)...")
    active_sgl, lbd_sgl, alpha_sgl = run_simon_sparse_group_lasso(X_gdelt_tilde, y_tilde, groups_array, secteurs, alpha=0.05)
    print(f"  -> Lambda optimal : {lbd_sgl:.4f} (avec Alpha fixe à {alpha_sgl})")
    
    if active_sgl.empty:
        print("  -> Résultat : Aucune variable conservée.")
    else:
        print(f"  -> {len(active_sgl)} variables conservées.")
        
        # --- ÉTAPE 5 : INFÉRENCE POST-LASSO OLS ---
        print("\n[ÉTAPE 5] Tribunal Statistique : Inférence Post-Lasso OLS sur les survivants...")
        post_lasso_df, r2_sgl, r2_adj_sgl = run_post_lasso_ols(X_gdelt_tilde, y_tilde, active_sgl)
        
        print(f"  -> R-squared (Pouvoir explicatif des médias purs) : {r2_sgl:.4f}")
        print(f"  -> R-squared Ajusté : {r2_adj_sgl:.4f}\n")
        print(post_lasso_df.to_string(index=False))
        
        final_results[region] = {
            'SGL_Selection': active_sgl,
            'OLS_Inference': post_lasso_df,
            'R2_adj': r2_adj_sgl
        }


 PIPELINE ACADÉMIQUE COMPLET : US
  ⚕️ Sauvetage par Winsorisation appliqué sur : att_weight_agriculture_infrastructure_tech
✓ ÉTAPE 0 & 1 : 19 variables stationnarisées. Les variables mères sont exclues.
✓ ÉTAPE 2 (FWL) : Matrice de 130 mois. Ljung-Box p-value = 0.2442

[ÉTAPE 3] Group Lasso (Méthode Yuan & Lin 2006, sélection via Cp)...
  -> Lambda optimal : 0.0026 (Critère Cp min : 73.26)
  -> 45 variables conservées.

[ÉTAPE 4] Sparse Group Lasso (Méthode Simon et al. 2013, sélection via CV avec KFold)...
  -> Lambda optimal : 0.0061 (avec Alpha fixe à 0.05)
  -> 25 variables conservées.

[ÉTAPE 5] Tribunal Statistique : Inférence Post-Lasso OLS sur les survivants...
  -> R-squared (Pouvoir explicatif des médias purs) : 0.3800
  -> R-squared Ajusté : 0.2309

                            Variable  Coefficient  P-value  CI_Lower  CI_Upper
                           Intercept       0.0000   1.0000   -0.0270    0.0270
   att_weight_commodities_extraction      -0.0308   0.1962   -0.0778

In [94]:
# ==============================================================================
# CELLULE 4 : EXÉCUTION DU PIPELINE COMPLET (AVEC INFÉRENCE ÉTAPE 5)
# ==============================================================================

CONFIG_REGIONS = {
    'UK': {
        'macro_vars': {
            'inflation': {'ticker': 'GBRCPIALLMINMEI', 'transform': 'pct_change'}, 
            'rate': {'ticker': 'IUMABEDR', 'transform': 'diff'},                   # Ticker CORRIGÉ (Fichier local)
            'unemp': {'ticker': 'LRHUTTTTGBM156S', 'transform': 'diff'},           
            'indpro': {'ticker': 'GBRPRINTO01GYSAM', 'transform': 'pct_change'},   
            'money': {'ticker': 'LPMBD93', 'transform': 'pct_change'},             # Ticker CORRIGÉ (Fichier local)
            'fx': {'ticker': 'XUDLUSS', 'transform': 'pct_change'},                # Ticker CORRIGÉ (Fichier local)
            'oil': {'ticker': 'DCOILBRENTEU', 'transform': 'pct_change'}           
        },
        'inflation_lags': [1, 2, 3, 6, 12],                                  # Lags corrigés pour l'autocorrélation
        'deseasonalize_y': False                                                   # Lissage par moyenne mobile désactivé
    }
}

final_results = {}

for region, config in CONFIG_REGIONS.items():
    print(f"\n" + "="*70)
    print(f" PIPELINE ACADÉMIQUE COMPLET : {region.upper()}")
    print(f"======================================================================")
    
    # --- PRÉPARATION MACRO ---
    df_macro = fetch_and_prep_macro_dynamic(config, START_DATE, END_DATE, region=region)
    df_macro = df_macro[df_macro.index >= MIN_INDEX_DATE]
    
    # --- PRÉPARATION GDELT ---
    cols_gdelt = get_filtered_gdelt_cols(df_geo)
    df_gdelt_raw = df_geo[df_geo['region_key'] == region].set_index('period')[cols_gdelt]
    
    # ---------------------------------------------------------
    # AMPUTATION CHIRURGICALE : On retire la variable asymétrique
    # ---------------------------------------------------------
    col_to_drop = 'att_weight_finance_international_orgs'
    if col_to_drop in df_gdelt_raw.columns:
        df_gdelt_raw = df_gdelt_raw.drop(columns=[col_to_drop])
    
    # --- STATIONNARISATION ---
    df_gdelt_stat, diff_count = strict_stationarize(df_gdelt_raw)
    print(f"✓ ÉTAPE 0 & 1 : {diff_count} variables stationnarisées. Les variables mères sont exclues.")
    
    # --- JOINTURE ---
    df_final = df_gdelt_stat.join(df_macro, how='inner')
    y_raw = df_final['inflation']
    
    if config['deseasonalize_y']:
        decomp = seasonal_decompose(y_raw, model='additive', period=12)
        y_corr = (y_raw - decomp.seasonal).dropna()
    else:
        y_corr = y_raw.dropna()
        
    X_full = df_final.drop(columns=['inflation'])
    y_corr, X_contemp = y_corr.align(X_full, join='inner')
    
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X_contemp), columns=X_contemp.columns, index=X_contemp.index)
    
    X_macro = X_scaled[[col for col in X_scaled.columns if col not in cols_gdelt]]
    X_gdelt = X_scaled[[col for col in X_scaled.columns if col in cols_gdelt]]
    
    # --- FWL ---
    y_tilde, X_gdelt_tilde = apply_fwl_orthogonalization(X_macro, X_gdelt, y_corr)
    pval = acorr_ljungbox(y_tilde, lags=[12], return_df=True)['lb_pvalue'].values[0]
    print(f"✓ ÉTAPE 2 (FWL) : Matrice de {len(y_tilde)} mois. Ljung-Box p-value = {pval:.4f}")
    
    secteurs = [col.split('_')[2] for col in X_gdelt_tilde.columns]
    sector_to_id = {sec: i for i, sec in enumerate(list(set(secteurs)))}
    groups_array = np.array([sector_to_id[sec] for sec in secteurs])
    
    # --- ÉTAPE 3 : GROUP LASSO (YUAN & LIN) ---
    print("\n[ÉTAPE 3] Group Lasso (Méthode Yuan & Lin 2006, sélection via Cp)...")
    active_gl, lbd_gl, cp_gl = run_yuan_lin_group_lasso(X_gdelt_tilde, y_tilde, groups_array, secteurs)
    print(f"  -> Lambda optimal : {lbd_gl:.4f} (Critère Cp min : {cp_gl:.2f})")
    print(f"  -> {len(active_gl)} variables conservées.")
        
    # --- ÉTAPE 4 : SPARSE GROUP LASSO (SIMON ET AL.) ---
    print("\n[ÉTAPE 4] Sparse Group Lasso (Méthode Simon et al. 2013, sélection via CV avec KFold)...")
    active_sgl, lbd_sgl, alpha_sgl = run_simon_sparse_group_lasso(X_gdelt_tilde, y_tilde, groups_array, secteurs, alpha=0.05)
    print(f"  -> Lambda optimal : {lbd_sgl:.4f} (avec Alpha fixe à {alpha_sgl})")
    
    if active_sgl.empty:
        print("  -> Résultat : Aucune variable conservée.")
    else:
        print(f"  -> {len(active_sgl)} variables conservées.")
        
        # --- ÉTAPE 5 : INFÉRENCE POST-LASSO OLS ---
        print("\n[ÉTAPE 5] Tribunal Statistique : Inférence Post-Lasso OLS sur les survivants...")
        post_lasso_df, r2_sgl, r2_adj_sgl = run_post_lasso_ols(X_gdelt_tilde, y_tilde, active_sgl)
        
        print(f"  -> R-squared (Pouvoir explicatif des médias purs) : {r2_sgl:.4f}")
        print(f"  -> R-squared Ajusté : {r2_adj_sgl:.4f}\n")
        print(post_lasso_df.to_string(index=False))
        
        final_results[region] = {
            'SGL_Selection': active_sgl,
            'OLS_Inference': post_lasso_df,
            'R2_adj': r2_adj_sgl
        }


 PIPELINE ACADÉMIQUE COMPLET : UK
✓ ÉTAPE 0 & 1 : 31 variables stationnarisées. Les variables mères sont exclues.
✓ ÉTAPE 2 (FWL) : Matrice de 120 mois. Ljung-Box p-value = 0.0599

[ÉTAPE 3] Group Lasso (Méthode Yuan & Lin 2006, sélection via Cp)...
  -> Lambda optimal : 0.0021 (Critère Cp min : 73.18)
  -> 50 variables conservées.

[ÉTAPE 4] Sparse Group Lasso (Méthode Simon et al. 2013, sélection via CV avec KFold)...
  -> Lambda optimal : 0.0076 (avec Alpha fixe à 0.05)
  -> 30 variables conservées.

[ÉTAPE 5] Tribunal Statistique : Inférence Post-Lasso OLS sur les survivants...
  -> R-squared (Pouvoir explicatif des médias purs) : 0.4693
  -> R-squared Ajusté : 0.2904

                                  Variable  Coefficient  P-value  CI_Lower  CI_Upper
                                 Intercept      -0.0000   1.0000   -0.0441    0.0441
              att_weight_agriculture_water      -0.0208   0.4861   -0.0799    0.0383
       att_weight_agriculture_agribusiness      -0.0575   0.17